# PDF 로드와 청킹 — 강사용 정답본

원본: `solution/Chapter01/03-rag/01_data_loading.py`  
대응 슬라이드: `99_pdf/Chapter01/day01/1-4-rag-theory-and-core-modules.pdf`

**학습 목표:** PDF → Document → 작은 청크와 overlap의 각 단계를 원본 코드에서 찾아 설명한다.

**실행 전:** 로컬 펫보험 PDF를 읽고 텍스트 청크를 만든다. PDF 파일 경로와 읽기 가능 여부를 확인한다.



## 흐름을 먼저 읽기

PDF → Document → 작은 청크와 overlap

핵심은 문서 준비·검색·답변 단계의 경계이다. 입력이 어디서 들어오고 결과가 어디로 이동하는지 화살표를 따라가자. 원본 주석과 TODO를 그대로 둔다. TODO는 CodeMate에게 작업을 지시하고 결과를 검토할 지점으로 읽는다.


## CodeMate 실습 지시

> 다음 코드 흐름을 순서대로 설명해 주세요: PDF → Document → 작은 청크와 overlap. 청크가 지나치게 작으면 어떤 문맥이 사라지는가? 원본 코드를 바꾸기 전에 근거가 되는 위치와 실행 위험을 알려 주세요.

CodeMate의 제안을 원본의 입력·출력·오류 흐름과 비교하고, 다른 결과가 나오면 다시 지시한다.


## 노트북 경로 설정

원본은 `__file__`로 PDF 위치를 찾는다. 노트북에서는 정의되지 않을 수 있으므로 원본 코드셀보다 먼저 이 설정 셀을 수동 실행한다. 프로젝트 루트를 찾지 못하면 명시적으로 오류가 난다. 원본 코드는 수정하지 않는다.


In [ ]:
from pathlib import Path
_cwd = Path.cwd().resolve()
_roots = [p for p in (_cwd, *_cwd.parents) if (p / 'pyproject.toml').is_file()]
if not _roots:
    raise FileNotFoundError('pyproject.toml이 있는 프로젝트 폴더에서 노트북을 열어 주세요.')
_project_root = _roots[0]
__file__ = str(_project_root / 'solution/Chapter01/03-rag/01_data_loading.py')
if not Path(__file__).is_file():
    raise FileNotFoundError(f'원본 Python 파일을 찾을 수 없습니다: {__file__}')


## 원본 코드 — 수동 실행

아래 셀은 `.py` 전체를 그대로 보존한다. 위에서 아래로 한 번에 실행되므로 환경과 부작용을 확인한다. **자동 실행/Run All을 사용하지 않는다.**

**이 코드셀의 실행 조건·부작용:** 로컬 펫보험 PDF를 읽고 텍스트 청크를 만든다. PDF 파일 경로와 읽기 가능 여부를 확인한다.



In [ ]:
"""
데이터 로딩과 청킹: PDF를 Document와 Chunk로 변환하기

핵심 개념:
  - Document = LangChain이 문서 본문과 메타데이터를 다루는 기본 단위
  - Chunking = 긴 문서를 검색 가능한 작은 조각으로 나누는 과정
  - chunk_size와 chunk_overlap은 검색 정확도와 문맥 보존 사이의 균형을 조정

실행:
  python 03-rag/01_data_loading.py
"""
from pathlib import Path

from langchain_community.document_loaders import PDFMinerLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


PDF_PATH = Path(__file__).parent / 'data' / 'meritz_pet_insurance.pdf'


# ── PDF 로딩 ─────────────────────────────────────────────────────────────────
# PDFMinerLoader: PDF 파일을 LangChain Document 목록으로 변환
# 각 Document는 page_content와 metadata를 가진다.
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()  # 결과는 list[Document]가 된다.


# ── 문서 청킹 ─────────────────────────────────────────────────────────────────
# 전체 문서는 한 번에 LLM에 넣기 어렵기 때문에 검색 가능한 Chunk로 나눈다.
# RecursiveCharacterTextSplitter: 문단 → 줄 → 단어 → 문자 순서로 분할 지점을 찾는다.
# chunk_size: 각 Chunk의 최대 길이
# chunk_overlap: 인접 Chunk 사이에 겹쳐 둘 길이
text_splitter = RecursiveCharacterTextSplitter(
    # 작을수록 검색 단위가 정밀하지만 전체 문맥이 줄어든다.
    chunk_size=500,
    # 클수록 문맥 연결성이 좋아지지만 중복과 비용이 늘어난다.
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)


# ── 결과 확인 ─────────────────────────────────────────────────────────────────
for index, chunk in enumerate(chunks[:3]):
    text = " ".join(chunk.page_content.split())
    print(f"{index}. len={len(chunk.page_content)} \n- {text[:120]}...")
    print()


## 결과를 검토하기

- 예상 동작: PDF에서 읽은 문서와 청크의 개수·일부 내용을 출력한다.
- 확인 질문: 청크가 지나치게 작으면 어떤 문맥이 사라지는가?
- 오류가 나면 먼저 원본 코드의 어느 단계인지 찾고, CodeMate가 제안한 수정의 이유와 부작용을 설명하게 한다.
